# VPN-Sentinel: Model Evaluation and Diagnostics

This notebook focuses on generating data and deeply evaluating the performance of the VPN classification models, including ROC curves and confusion matrices.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
plt.style.use('seaborn-v0_8')


## 1. Data Generation
We will generate the synthetic network flows if they don't already exist.

In [ ]:
from data_generator import generate_flows
train_path = DATA_DIR / 'train_flows.csv'
test_path = DATA_DIR / 'test_flows.csv'
if not (train_path.exists() and test_path.exists()):
    print('Generating new flow data...')
    df = generate_flows(15000)
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['is_vpn'])
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)
else:
    print('Loading existing data...')
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')


## 2. Model Training

In [ ]:
features = [
    'duration', 'fwd_pkt_len_mean', 'bwd_pkt_len_mean',
    'flow_iat_mean', 'flow_iat_std', 'jitter_ratio', 'packets_per_sec', 'bytes_per_sec'
]
X_train = train_df[features].fillna(-1)
y_train = train_df['is_vpn']
X_test = test_df[features].fillna(-1)
y_test = test_df['is_vpn']
clf = RandomForestClassifier(n_estimators=150, max_depth=15, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred))


## 3. Evaluation Diagnostics
Visualizing Confusion Matrix and ROC Curve

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0])
ax[0].set_title('Confusion Matrix')
ax[0].set_xlabel('Predicted')
ax[0].set_ylabel('True')
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
ax[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
ax[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax[1].set_xlim([0.0, 1.0])
ax[1].set_ylim([0.0, 1.05])
ax[1].set_title('Receiver Operating Characteristic')
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate')
ax[1].legend(loc="lower right")
plt.tight_layout()
plt.show()
